> **Dependencies**: TensorFlow ≥ 2.12, TensorFlow-Probability ≥ 0.20. Installed by the first cell.
>
> **Codebase**: requires the `advanced_particle_filter` codebase to be uploaded as a zip and extracted at `/content/advanced_particle_filter/`. The first setup cell handles this.
>
> **What it does**: Three-phase MAP + Laplace + preconditioned-HMC pipeline.

---

# SVSSM: Find-Mode-Then-Sample Pipeline

Bayesian calibration of a 2-asset Stochastic Volatility State-Space Model (SVSSM) using a three-phase pipeline:

1. **Phase 1 — MAP via Adam + DPF-soft**: gradient-based search for the posterior mode using a differentiable particle filter with soft resampling. Adam's second-moment normalization handles the DPF gradient-magnitude attenuation.
2. **Phase 2 — Laplace covariance via FD Hessian**: build a finite-difference Hessian at the MAP, eigendecompose, and obtain local curvature. When indefinite (saddle), spectral regularization bounds the condition number.
3. **Phase 3 — Preconditioned HMC with |H|-mass-matrix**: HMC in native parameter space using `M = |H|` as the mass matrix. Absorbs the sign of any indefinite Hessian via `|·|`, preserves the Hessian's eigenvectors (correlation structure) while using eigenvalue magnitudes for per-direction step sizes.

## Summary of findings

| Metric | Raw HMC-DPF baseline | This pipeline (T=300) |
|---|---|---|
| Wall clock | hours | ~60 min on A100 |
| Phase 3 accept rate | ~67% | ~62% |
| R-hat | ~1.1 | 1.0–1.3 |
| Phase 2 Hessian structure | n/a | Positive-definite at T=300 |

The posterior mode at T=300 sits at parameters that produce observationally similar but quantitatively distinguishable trajectories compared to ground truth. This is a finite-sample feature of the posterior geometry under the chosen informative prior — not a pipeline error. See section 6 for the direct verification.

## How to replicate

1. Run cells 0 through 3 (setup + package upload).
2. Run cell 5 (smoke test) to confirm plumbing.
3. Run cell 7 (full pipeline, ~60 min on A100 40GB).
4. Run cells 9–16 for diagnostic plots and the final posterior-peak verification.

---
## 0. Setup

In [ ]:
!pip install -q tensorflow-probability

import tensorflow as tf
import tensorflow_probability as tfp
import numpy as np
import matplotlib.pyplot as plt

print('TF', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
import os, zipfile
from google.colab import files

# Upload the advanced_particle_filter.zip produced by the latest build
uploaded = files.upload()
zip_name = [k for k in uploaded if k.endswith('.zip')][-1]

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

# Sanity check: module layout
import advanced_particle_filter
print('Package loaded from:', advanced_particle_filter.__file__)

---
## 1. Smoke test (tiny config)

Verifies pipeline plumbing — not for interpretation. Takes <1 minute.

In [ ]:
from advanced_particle_filter.mle.smoke_test import smoke
_ = smoke()

---
## 2. Full pipeline run (T=300, N=500)

Config: soft DPF at N=500, 4 Adam restarts × 2000 steps, preconditioned HMC at 4 chains.

Memory: per-step Gumbel noise generation (not pre-allocated) keeps peak GPU memory at ~8 GB even at T=300.

Expected wall clock on A100 40 GB: ~60 minutes total (Phase 1 ~25 min, Phase 2 ~2 min, Phase 3 ~35 min).

In [ ]:
from advanced_particle_filter.mle.run_mle_laplace import main

results = main(
    T=300,
    data_seed=0,
    n_particles=500,
    alpha=0.5,
    # Phase 1
    B_restart=4,
    adam_steps=2000,
    adam_lr=0.003,
    adam_n_mc=16,
    adam_jitter=0.1,
    # Phase 2
    laplace_eps=0.05,
    laplace_n_seeds=15,
    laplace_chunk_size=8,
    laplace_kappa_target=10.0,
    # Phase 3 — preconditioned HMC
    run_phase3=True,
    phase3_method='preconditioned',
    hmc_B_chain=4,
    hmc_burnin=50,
    hmc_samples=200,
    hmc_leapfrog=10,
    hmc_step=0.3,
    hmc_n_mc=2,
    include_prior=True,
)

---
## 3. Phase 1 diagnostics — Adam convergence

Check each restart's trajectory. At T=300:
- Loss (−log posterior) should drop from ~870 at init to a plateau around ~858
- Gradient norm should drop by 10–100× from its initial spike and plateau at ~1 (DPF noise floor)
- All 4 restarts should cluster at similar final log-posteriors (indicates unique mode, no multi-modality)

In [ ]:
adam = results['adam']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
for r in range(adam.loss_history.shape[1]):
    ax.plot(-adam.loss_history[:, r], alpha=0.8, lw=0.9, label=f'restart {r}')
ax.set_xlabel('Adam step'); ax.set_ylabel('log-posterior')
ax.set_title('Loss per restart')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1]
for r in range(adam.grad_norm_history.shape[1]):
    ax.plot(adam.grad_norm_history[:, r], alpha=0.8, lw=0.9, label=f'restart {r}')
ax.set_yscale('log')
ax.set_xlabel('Adam step'); ax.set_ylabel('||grad||')
ax.set_title('Gradient norm per restart')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f'\nFinal log-posteriors per restart: {adam.final_log_post.numpy().round(2)}')
print(f'Best restart: {adam.best_restart}')

---
## 4. Phase 2 diagnostics — Laplace covariance structure

At T=300 we expect:
- **All eigenvalues of −H positive** (MAP is a proper mode, not a saddle)
- **0/9 eigenvalues clipped** by spectral regularization (no need for regularization)
- Hessian shows strong diagonal curvature with meaningful off-diagonal correlation
- Some entries may have high relative SE (DPF noise at n_seeds=15) — that's okay; the eigenvalue structure is the primary diagnostic

In [ ]:
lap = results['laplace']
Sigma = lap.Sigma_laplace.numpy()
H = lap.H.numpy()
H_se = lap.H_se.numpy()

param_names = ['mu_0', 'log(gap)', 'Phi_00', 'Phi_01', 'Phi_10', 'Phi_11',
               'log(L_11)', 'log(L_22)', 'L_21']

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))

# Laplace correlation
ax = axes[0]
std = np.sqrt(np.diag(Sigma))
corr = Sigma / (std[:, None] * std[None, :])
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(9)); ax.set_yticks(range(9))
ax.set_xticklabels(param_names, rotation=45, ha='right')
ax.set_yticklabels(param_names)
ax.set_title('Laplace posterior correlation')
plt.colorbar(im, ax=ax, fraction=0.045)

# Hessian
ax = axes[1]
vmax = np.max(np.abs(H))
im = ax.imshow(H, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
ax.set_xticks(range(9)); ax.set_yticks(range(9))
ax.set_xticklabels(param_names, rotation=45, ha='right')
ax.set_yticklabels(param_names)
ax.set_title('Hessian of log-posterior')
plt.colorbar(im, ax=ax, fraction=0.045)

# Relative SE
ax = axes[2]
rel_se = np.abs(H_se) / (np.abs(H) + 1e-12)
im = ax.imshow(np.log10(rel_se + 1e-6), cmap='viridis')
ax.set_xticks(range(9)); ax.set_yticks(range(9))
ax.set_xticklabels(param_names, rotation=45, ha='right')
ax.set_yticklabels(param_names)
ax.set_title('log10(SE/|H|) — noise per entry')
plt.colorbar(im, ax=ax, fraction=0.045)
plt.tight_layout(); plt.show()

print(f'\nEigenvalues of -H: {lap.eigenvalues_negH.numpy().round(3)}')
print(f'Eigvals clipped: {lap.n_eigvals_clipped}/9 (kappa_target={lap.kappa_target:.1f})')
print(f'Max SE/|H|: {rel_se.max():.2f} (increase n_seeds if > 0.5)')

---
## 5. Phase 3 diagnostics — HMC sample covariance

In preconditioned HMC, the mass matrix M = |H| encodes expected curvature. We transform samples by L_M^{-1} (the 'whitened' analog); if the posterior really is locally Gaussian-matching-H, this should produce samples with variance ≈ 1 in each direction.

At T=300, Laplace and HMC marginal stds should agree within a factor of ~2–5. Any remaining gap reflects non-Gaussianity in the posterior that Laplace can't capture.

In [ ]:
w = results['whitened']
if w is None:
    print('Phase 3 was skipped (run_phase3=False).')
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    
    # Whitened sample covariance
    ax = axes[0]
    flat_white = w.samples_z_white.reshape(-1, 9)
    cov_white = np.cov(flat_white, rowvar=False)
    im = ax.imshow(cov_white - np.eye(9), cmap='RdBu_r', vmin=-1.5, vmax=1.5)
    ax.set_xticks(range(9)); ax.set_yticks(range(9))
    ax.set_xticklabels(param_names, rotation=45, ha='right')
    ax.set_yticklabels(param_names)
    ax.set_title('Sample cov (whitened) − I')
    plt.colorbar(im, ax=ax, fraction=0.045)
    
    # Laplace vs HMC marginal std
    ax = axes[1]
    std_lap = np.sqrt(np.diag(Sigma))
    std_hmc = w.samples_z.reshape(-1, 9).std(axis=0)
    x = np.arange(9)
    ax.bar(x - 0.2, std_lap, width=0.4, label='Laplace', alpha=0.8)
    ax.bar(x + 0.2, std_hmc, width=0.4, label='HMC', alpha=0.8)
    ax.set_xticks(x); ax.set_xticklabels(param_names, rotation=45, ha='right')
    ax.set_ylabel('std')
    ax.set_title('Marginal std: Laplace vs HMC')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    
    print(f'\nR-hat (orig coords):  {w.rhat.round(3)}')
    print(f'R-hat (white coords): {w.rhat_white.round(3)}')
    print(f'Accept rate: {w.accept_rate:.2%}')

---
## 6. Point estimates vs ground truth

Report the MAP point estimate and HMC posterior summary, compared to the data-generating parameters. At T=300 we expect some parameters (especially μ) to clearly move toward truth, while others (Φ_00, Σ_η[0,0]) remain biased due to finite-sample posterior geometry + prior influence. The next section verifies this is a property of the posterior itself, not a pipeline error.

In [ ]:
from advanced_particle_filter.hmc.parameterization import unpack_batched

truth = results['truth']
z_hat = adam.z_hat.numpy()
p_hat = unpack_batched(tf.constant(z_hat[np.newaxis, :], dtype=tf.float64))

print('=' * 60)
print('  MAP point estimates vs truth')
print('=' * 60)
print(f'\nmu_hat   = {p_hat.mu[0].numpy().round(3)}')
print(f'mu_true  = {truth["mu"].numpy()}')
print(f'\nPhi_hat  =\n{p_hat.Phi[0].numpy().round(3)}')
print(f'Phi_true =\n{truth["Phi"].numpy()}')
Sigma_hat = p_hat.Sigma_eta_chol[0].numpy() @ p_hat.Sigma_eta_chol[0].numpy().T
print(f'\nSigma_hat  =\n{Sigma_hat.round(4)}')
print(f'Sigma_true =\n{truth["Sigma_eta"].numpy()}')

if w is not None:
    flat = w.samples_z.reshape(-1, 9)
    phi00_samples = flat[:, 2]
    print('\n' + '=' * 60)
    print('  Phi_00  (raw HMC gave ~0.55 vs truth 0.85)')
    print('=' * 60)
    print(f'  MAP:  {z_hat[2]:.3f}')
    print(f'  HMC mean: {phi00_samples.mean():.3f}  std: {phi00_samples.std():.3f}')
    print(f'  HMC 95% CI: [{np.percentile(phi00_samples, 2.5):.3f}, '
          f'{np.percentile(phi00_samples, 97.5):.3f}]')
    print(f'  TRUTH: 0.850')

---
## 7. Is z_hat really the posterior peak?

The decisive check that the pipeline is correct and any bias is intrinsic: evaluate the DPF-based log-posterior at `z_hat` and at `z_truth`, averaging over many PF seeds for low variance.

- If `log p(y|z_hat) + log p(z_hat)` > `log p(y|z_truth) + log p(z_truth)` → **the posterior's peak really is at z_hat (or nearby)**. The pipeline correctly identified it. Any discrepancy from truth is a finite-sample + prior-choice artifact, not an algorithm error.
- If instead `z_truth` has higher log-posterior → Adam landed in the wrong basin and the pipeline has a real bug.

We also run a 1D profile over Φ_00 (holding other params at truth) to visualize where the posterior peak sits in that single direction.

In [ ]:
from advanced_particle_filter.tf_filters import TFDifferentiableParticleFilter
from advanced_particle_filter.hmc.parameterization import (
    unpack_batched, log_prior_batched, TOTAL_DIM,
)

DTYPE = tf.float64
y_obs = results['y_obs']
T_data = y_obs.shape[0]

# DPF for high-N log-posterior evaluation (no gradient needed, so we can
# push N higher than in training)
dpf_eval = TFDifferentiableParticleFilter(
    n_particles=1000, resampler='soft', alpha=0.5, dtype=DTYPE,
)

# Build z_truth (same layout as parameterization.py)
mu_t  = truth['mu'].numpy()
Phi_t = truth['Phi'].numpy()
L_t   = np.linalg.cholesky(truth['Sigma_eta'].numpy())
z_truth = np.array([
    mu_t[0], np.log(mu_t[1] - mu_t[0]),
    Phi_t[0, 0], Phi_t[0, 1], Phi_t[1, 0], Phi_t[1, 1],
    np.log(L_t[0, 0]), np.log(L_t[1, 1]), L_t[1, 0],
])


def eval_logpost_dpf_mean(z_batch, n_seeds=20, seed_base=90000):
    """Average DPF log-posterior over n_seeds PF realizations."""
    z_t = tf.constant(z_batch, dtype=DTYPE)
    values = np.zeros((n_seeds, z_batch.shape[0]))
    for s in range(n_seeds):
        rng = tf.random.Generator.from_seed(seed_base + s)
        params = unpack_batched(z_t)
        ll = dpf_eval.filter(params, tf.constant(y_obs, dtype=DTYPE), rng).log_evidence
        lp = log_prior_batched(z_t)
        values[s] = (ll + lp).numpy()
    return values.mean(axis=0), values.std(axis=0) / np.sqrt(n_seeds)


# ---- Point comparison ----
print(f'Evaluating log-posterior at z_hat and z_truth (T={T_data}, 20 PF seeds at N=1000)...')
z_pair = np.stack([z_hat, z_truth])
lp_mean, lp_se = eval_logpost_dpf_mean(z_pair, n_seeds=20)
print(f'\n  log_post(z_hat)   = {lp_mean[0]:+.3f} ± {lp_se[0]:.3f}')
print(f'  log_post(z_truth) = {lp_mean[1]:+.3f} ± {lp_se[1]:.3f}')
diff = lp_mean[0] - lp_mean[1]
print(f'  diff (hat - truth) = {diff:+.3f}')
if diff > 3 * max(lp_se):
    print(f'\n  -> z_hat has DECISIVELY higher log-posterior than truth.')
    print(f'     Pipeline correctly identified the mode; bias w.r.t. truth is intrinsic.')
elif diff > 0:
    print(f'\n  -> z_hat has nominally higher log-posterior than truth (within noise).')
else:
    print(f'\n  -> z_truth has higher log-posterior than z_hat!')
    print(f'     This would indicate Adam missed the true MAP basin.')

# ---- 1D profile over Phi_00 ----
print(f'\nProfile sweep over Phi_00 (holding others at z_truth)...')
phi00_grid = np.linspace(0.30, 0.95, 14)
z_profile = np.tile(z_truth[None, :], (len(phi00_grid), 1))
z_profile[:, 2] = phi00_grid

lp_profile, se_profile = eval_logpost_dpf_mean(z_profile, n_seeds=20, seed_base=91000)

fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(phi00_grid, lp_profile, yerr=se_profile, marker='o',
            label='log p(y|Phi_00, rest=truth) + log p(Phi_00)')
ax.axvline(0.85, color='green', linestyle='--', label='truth Phi_00 = 0.85')
ax.axvline(z_hat[2], color='red', linestyle='--', label=f'z_hat Phi_00 = {z_hat[2]:.2f}')
ax.axhline(lp_mean[0], color='red', linestyle=':', alpha=0.5,
           label=f'log_post(z_hat) = {lp_mean[0]:.1f}')
ax.axhline(lp_mean[1], color='green', linestyle=':', alpha=0.5,
           label=f'log_post(z_truth) = {lp_mean[1]:.1f}')
ax.set_xlabel('Phi_00'); ax.set_ylabel('log-posterior (mean of 20 PF seeds)')
ax.set_title(f'Profile log-posterior vs Phi_00 at T={T_data}')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

peak_idx = np.argmax(lp_profile)
truth_idx = np.argmin(np.abs(phi00_grid - 0.85))
print(f'\nProfile peak at Phi_00 = {phi00_grid[peak_idx]:.3f}, '
      f'log_post = {lp_profile[peak_idx]:+.3f}')
print(f'log_post at Phi_00 = 0.85 (truth): {lp_profile[truth_idx]:+.3f}')
print(f'Profile peak is {lp_profile[peak_idx] - lp_profile[truth_idx]:+.3f} '
      f'log-units above truth.')

---
## 8. Trajectory diagnostic

Visualize how similar the simulated trajectories are from `z_hat` vs `z_truth` on the same noise seeds. Uses hardcoded parameters from the T=300 pipeline result — standalone from `results` so you can reproduce without re-running.

**Interpretation:** if z_hat-simulated and z_truth-simulated trajectories produce similar amplitude ranges and volatility-clustering patterns, the two parameter sets are observationally similar. Small differences in means and variances confirm they are not *identical* — but both are plausible fits to the observed data, which is exactly what the posterior-peak verification in section 7 is meant to show.

In [ ]:
from advanced_particle_filter.tf_models.svssm import simulate_svssm
from advanced_particle_filter.hmc.run_hmc_poc import make_contagion_data
from scipy.linalg import solve_discrete_lyapunov

DTYPE = tf.float64
T_sim = 300
data_seed = 0

# ==== Hardcoded θ_hat from the T=300 pipeline run (data_seed=0) ====
mu_hat_np  = np.array([-1.021, 0.716], dtype=np.float64)
Phi_hat_np = np.array([[0.603, 0.249],
                       [0.077, 0.826]], dtype=np.float64)
Sigma_hat_np = np.array([[0.3089, 0.0002],
                         [0.0002, 0.2170]], dtype=np.float64)
Lchol_hat_np = np.linalg.cholesky(Sigma_hat_np)

mu_true_np  = np.array([-1.0, 0.5], dtype=np.float64)
Phi_true_np = np.array([[0.85, 0.12],
                        [0.02, 0.90]], dtype=np.float64)
Sigma_true_np = np.array([[0.0225, 0.018],
                          [0.018,  0.160]], dtype=np.float64)
Lchol_true_np = np.linalg.cholesky(Sigma_true_np)

mu_hat_tf     = tf.constant(mu_hat_np,    dtype=DTYPE)
Phi_hat_tf    = tf.constant(Phi_hat_np,   dtype=DTYPE)
Lchol_hat_tf  = tf.constant(Lchol_hat_np, dtype=DTYPE)
mu_true_tf    = tf.constant(mu_true_np,    dtype=DTYPE)
Phi_true_tf   = tf.constant(Phi_true_np,   dtype=DTYPE)
Lchol_true_tf = tf.constant(Lchol_true_np, dtype=DTYPE)

# Regenerate the observed data (same seed used in the pipeline)
_, y_obs_tf, _ = make_contagion_data(T=T_sim, seed=data_seed)
y_obs_ref = y_obs_tf.numpy()

# Paired simulations (same RNG seed per replicate for both hat and truth)
N_rep = 5
h_hat  = np.zeros((N_rep, T_sim, 2)); y_hat  = np.zeros((N_rep, T_sim, 2))
h_true = np.zeros((N_rep, T_sim, 2)); y_true = np.zeros((N_rep, T_sim, 2))
for rep in range(N_rep):
    rng_h = tf.random.Generator.from_seed(20000 + rep)
    rng_t = tf.random.Generator.from_seed(20000 + rep)
    h_h, y_h = simulate_svssm(mu_hat_tf,  Phi_hat_tf,  Lchol_hat_tf,  T_sim, rng_h)
    h_t, y_t = simulate_svssm(mu_true_tf, Phi_true_tf, Lchol_true_tf, T_sim, rng_t)
    h_hat[rep]  = h_h.numpy(); y_hat[rep]  = y_h.numpy()
    h_true[rep] = h_t.numpy(); y_true[rep] = y_t.numpy()

# Plot: 4 rows x 2 assets
fig, axes = plt.subplots(4, 2, figsize=(14, 12))
t_axis = np.arange(T_sim)
for d in range(2):
    # Row 0: h from z_hat
    ax = axes[0, d]
    for r in range(N_rep):
        ax.plot(t_axis, h_hat[r, :, d], color='C1', alpha=0.5, lw=1)
    ax.plot(t_axis, h_hat.mean(0)[:, d],  color='C1', lw=2, label='z_hat')
    ax.plot(t_axis, h_true.mean(0)[:, d], color='C2', lw=2, label='z_truth (mean)')
    ax.set_title(f'h_t asset {d+1}: z_hat (orange) vs truth mean (green)')
    ax.legend(); ax.grid(alpha=0.3)
    
    # Row 1: h from z_truth
    ax = axes[1, d]
    for r in range(N_rep):
        ax.plot(t_axis, h_true[r, :, d], color='C2', alpha=0.5, lw=1)
    ax.plot(t_axis, h_true.mean(0)[:, d], color='C2', lw=2, label='z_truth')
    ax.plot(t_axis, h_hat.mean(0)[:, d],  color='C1', lw=2, label='z_hat (mean)')
    ax.set_title(f'h_t asset {d+1}: z_truth (green) vs hat mean (orange)')
    ax.legend(); ax.grid(alpha=0.3)
    
    # Row 2: y from z_hat vs observed
    ax = axes[2, d]
    for r in range(N_rep):
        ax.plot(t_axis, y_hat[r, :, d], color='C1', alpha=0.4, lw=0.8)
    ax.plot(t_axis, y_obs_ref[:, d], color='k', lw=1.2, alpha=0.7, label='observed')
    ax.set_title(f'y_t asset {d+1}: z_hat (orange) vs observed (black)')
    ax.legend(); ax.grid(alpha=0.3)
    
    # Row 3: y from z_truth vs observed
    ax = axes[3, d]
    for r in range(N_rep):
        ax.plot(t_axis, y_true[r, :, d], color='C2', alpha=0.4, lw=0.8)
    ax.plot(t_axis, y_obs_ref[:, d], color='k', lw=1.2, alpha=0.7, label='observed')
    ax.set_title(f'y_t asset {d+1}: z_truth (green) vs observed (black)')
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Summary stats
print('\n' + '=' * 70)
print('  Summary: observed vs z_hat-simulated vs z_truth-simulated')
print('=' * 70)
for d in range(2):
    yh = y_hat[:, :, d].flatten()
    yt = y_true[:, :, d].flatten()
    yo = y_obs_ref[:, d]
    print(f'\nAsset {d+1}:')
    print(f'  observed: mean={yo.mean():+.3f}  std={yo.std():.3f}')
    print(f'  z_hat:    mean={yh.mean():+.3f}  std={yh.std():.3f}')
    print(f'  z_truth:  mean={yt.mean():+.3f}  std={yt.std():.3f}')

# Analytical stationary variance of h
Var_h_hat  = solve_discrete_lyapunov(Phi_hat_np,  Sigma_hat_np)
Var_h_true = solve_discrete_lyapunov(Phi_true_np, Sigma_true_np)
print('\n' + '=' * 70)
print('  Implied stationary Var(h) (analytical, from discrete Lyapunov eq.)')
print('=' * 70)
print(f'  z_hat  diag: {np.diag(Var_h_hat).round(3)}')
print(f'  z_true diag: {np.diag(Var_h_true).round(3)}')